In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
import subprocess
subprocess.run(["pip", "install", "transformers", "datasets", "peft", "trl", "bitsandbytes", "accelerate", "-q"])

import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
print(f"GPU 0: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00
CUDA available: True
GPU count: 1
GPU 0: Tesla T4


In [3]:
DATASET_PATH = "/kaggle/input/datasets/sarveshwaran160405/cleanoapair/dataset_clean.jsonl"

pairs = []
with open(DATASET_PATH, "r") as f:
    for line in f:
        pairs.append(json.loads(line))

print(f"Total pairs: {len(pairs)}")

Total pairs: 2620


In [4]:
def format_pair(pair):
    return {"text": f"### Instruction:\n{pair['instruction']}\n\n### Response:\n{pair['response']}"}

formatted = [format_pair(p) for p in pairs]
dataset = Dataset.from_list(formatted)
dataset = dataset.train_test_split(test_size=0.05, seed=42)
print(f"Train: {len(dataset['train'])} | Test: {len(dataset['test'])}")

Train: 2489 | Test: 131


In [5]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="cuda:0",
)

model = prepare_model_for_kbit_training(model)
print("Model loaded!")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded!


In [6]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [7]:
training_args = SFTConfig(
    output_dir="/kaggle/working/k8s-tinyllama",
    num_train_epochs=7,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    dataset_text_field="text",
    max_length=512,
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)

print("Starting training...")
trainer.train()
print("Training complete!")

Adding EOS to train dataset:   0%|          | 0/2489 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2489 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2489 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/131 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/131 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/131 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting training...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,1.759494,1.746718,1.739332,195895.000000,0.604327
400,1.606168,1.713277,1.654649,391466.000000,0.606607
600,1.482680,1.707320,1.551105,587574.000000,0.615719
800,1.343540,1.754182,1.484455,783075.000000,0.610759
1000,1.275984,1.774996,1.456701,978916.000000,0.609589


Training complete!


In [8]:
OUTPUT_PATH = "/kaggle/working/k8s-tinyllama-final"
trainer.model.save_pretrained(OUTPUT_PATH)
tokenizer.save_pretrained(OUTPUT_PATH)
print(f"Saved to {OUTPUT_PATH}")

Saved to /kaggle/working/k8s-tinyllama-final


In [9]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=150,
    device="cuda:0",
)

for query in ["Why is my pod in CrashLoopBackOff?", "What causes OOMKilled errors?"]:
    prompt = f"### Instruction:\n{query}\n\n### Response:\n"
    result = pipe(prompt, do_sample=False)
    print(f"Q: {query}")
    print(f"A: {result[0]['generated_text'].replace(prompt, '').strip()[:200]}")
    print()

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please 

Q: Why is my pod in CrashLoopBackOff?
A: The container in the Pod is in a CrashLoopBackOff state.

Q: What causes OOMKilled errors?
A: The container running out of memory.

